In [11]:
import xgboost as xgb
from pathlib import Path
print(xgb.__version__)

3.0.4


In [6]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

d:\VS CODE\Data Science\House Regression\.venv\Scripts\python.exe
3.0.4
d:\VS CODE\Data Science\House Regression\.venv\Lib\site-packages\xgboost\__init__.py


In [7]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [8]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("D:/VS CODE/Data Science/House Regression/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("D:/VS CODE/Data Science/House Regression/data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [9]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [12]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlruns_path = Path("D:/VS CODE/Data Science/House Regression/mlruns")

mlruns_path.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(mlruns_path.as_uri())
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/11/02 17:38:08 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-11-02 17:38:08,108] A new study created in memory with name: no-name-4845a3f5-3f73-4f3e-b99f-15ed55c12a92
[I 2026-11-02 17:38:32,173] Trial 0 finished with value: 76825.40921337031 and parameters: {'n_estimators': 955, 'max_depth': 5, 'learning_rate': 0.01293298412048036, 'subsample': 0.7651737376787336, 'colsample_bytree': 0.6697945589634582, 'min_child_weight': 4, 'gamma': 2.6162727411080997, 'reg_alpha': 8.145327521772423e-07, 'reg_lambda': 2.0593055076413726e-06}. Best is trial 0 with value: 76825.40921337031.
[I 2026-11-02 17:38:53,181] Trial 1 finished with value: 71216.03110630323 and parameters: {'n_estimators': 424, 'max_depth': 9, 'learning_rate': 0.039064424993960664, 'subsample': 0.6752573243994883, 'colsample_bytree': 0.6832606997633577, 'min_child_weight': 6, 'gamma': 3.2788247900791907, 'reg_alpha': 0.014786574012178706, 'reg_l

Best params: {'n_estimators': 341, 'max_depth': 9, 'learning_rate': 0.0248926970994545, 'subsample': 0.5007097793138255, 'colsample_bytree': 0.6139339273699723, 'min_child_weight': 8, 'gamma': 4.102433596586127, 'reg_alpha': 0.09276750626201871, 'reg_lambda': 1.1623193205826177e-05}


In [13]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 31679.42205136662
RMSE: 69957.03689037767
R²: 0.9621798851369915


d:\VS CODE\Data Science\House Regression\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [17:43:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/11/02 17:43:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/02 17:43:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
